In [1]:
from arcgis.gis import GIS
from arcgis.gis.admin import AGOLAdminManager

# Administering Your GIS Organizations Using ArcGIS API for Python

## Overview

- The ArcGIS ecosystem is vast
- Organizations can have multiple versions of any product or multiple products to manage
- How do you manage this?

## The Way of the Python

- The Python API allows administrators to manage, update and control what happens on your server
- Script from your favorite IDE or Notebook environment
- Cross platform support

### What can we do With ArcGIS Online?

## Getting Started

### Understand the `GIS` Object

The `GIS` object is the way users connect to ArcGIS Online and/or Enterprise

- It doesn't matter if you are an administrator of a user, we must start here.

#### Connecting to you `GIS` 

The ArcGIS API for Python support multiple ways of connecting to the `GIS`, which is ArcGIS Online or ArcGIS Enterprise

##### Anonymously

In [2]:
import pandas as pd
from arcgis.gis import GIS
gis = GIS() #anonymous connection

##### Built-In

- username/password login method
- usersname are case sensitive 

```python
gis = GIS(username='fakeaccount', password='fakepassword')
gis = GIS(url="https://www.mysite.com/portal", username='fakeaccount', password='fakepassword')
```

**Protecting Built-In Credentials**

- using `profiles` will help protect username and passwords.  
- prevents accidental sharing

1. Create a `GIS` object with the extra `profile` parameter

```python
gis = GIS(url="https://www.mysite.com/portal", 
          username='fakeaccount', 
          password='fakepassword', 
          profile='portal_profile')
```

2. Now connect using the `profile`

```python
gis = GIS(profile='portal_profile')
```

**What Happened?**

Instead of keeping your password in plain text, now we leverage the operating system's credential store for the logged in user.  The credentials never get passed on when you use profiles.

## Developer Credentials

- There are two types of credentials:
    - API Keys - provide a long lived token with a set of privileges or actions
    - Application Authentication - a limited set of privileges scopes to a given web application normally

#### Creating API Keys


In [3]:
from arcgis.gis import GIS
gis = GIS(profile='your_online_profile', trust_env=True)
gis.users.me

<User username:nparavicini_geosaurus>

##### Access the Administration Endpoint

In [4]:
from arcgis.gis.admin import AGOLAdminManager
from arcgis.gis.admin._stokenmgr import  TokenPrivilege
import datetime as _dt
admin:AGOLAdminManager = gis.admin
dev_creds = admin.developer_credentials

##### Create a Scoped API Key

In this scenerio, we are going to create an API key to allow a user to login via a token to view and create items.  We will define the privilege scope and set an expiration.

- Provide a `title`, `privileges`, `expiration` and `referer`
- The `expiration` can't be more than 1 year from now

In [5]:
api_credential = dev_creds.create(title='my first api key',
                                   privileges=[TokenPrivilege.PORTAL_ADMIN_VIEWITEMS, 
                                               TokenPrivilege.PORTAL_USER_CREATEITEM],
                                   referers=['http'],
                                   expiration=_dt.datetime.now() + _dt.timedelta(weeks=20))
api_credential

Creating an empty item.


<DeveloperCredential Item ID:"a204d54b408c493baebdbaf8afa9b82b">

In [6]:
token = api_credential.generate_token(slot=1)
token

{'access_token': 'AAPTxy8BH1VEsoebNVZXo8HurC_XeT3pOmZHnOyc93cCDQQRmt1hI7s2G8o8acsnlsV0Fpsxm8vp7Qmm6-gch2SBTF9ZJ_PEKeM6UcBeoX2Rp0PTbHFx9o5PpjmwiOly0eR1MjYRXAymWSBiQJhZsU9_nGjWK4xiyBtSDnyOfCHRRmAKA_sn641rCugkSkFnepedBcGclNlT3E1MPQYkEiATiLZKOeWdKel7f-tQDmtfOGM.AT1_PwBfTYt8',
 'expires_in': 15721139}

In [7]:
GIS(token=token['access_token']).users.me

<User username:nparavicini_geosaurus>

In [8]:
api_credential.delete()

True

## User Management

Users fuel your system. As an administrator your job is to ensure they can put up there content and know the site is reliable and safe.  The Python API is a tool to do just that!

In [9]:
from arcgis.gis import GIS
gis = GIS(profile='your_online_profile', verify_cert=False)

Setting `verify_cert` to False is a security risk, use at your own risk.


### Working with Existing Users

In [10]:
um = gis.users
um

< UserManager at https://geosaurus.maps.arcgis.com >

In [11]:
um

Type:        UserManager
String form: < UserManager at https://geosaurus.maps.arcgis.com >
File:        ~/miniconda3/envs/dec15_daily/lib/python3.13/site-packages/arcgis/gis/__init__.py
Docstring:  
The ``UserManager`` class is a helper class for managing GIS users. This class is not created by users directly.
An instance of this class, called 'users', is available as a property of the Gis object.
Users call methods on this 'users' object to manipulate (create, get, search, etc) users.

#### Search for Users

In [12]:
users = um.search("geo*")
users

[<User username:geosaurus_hub>,
 <User username:geosaurusaccnt_geosaurus>,
 <User username:nbda_user_50b0>,
 <User username:nbda_user_51d1>,
 <User username:nbda_user_9c6a>,
 <User username:nbda_user_a720>,
 <User username:nbda_user_eb59>]

### Advanced Search for Users

- Advanced Search gives your the full control to look up users.
- When using this search, nothing is there to guide you, so the queries are made by the end user.
- Provides the ability to get user counts and return users as dictionaries

##### How Many Users are in your Organization?

In [13]:
um.advanced_search(f"accountid:{gis.properties.id}", return_count=True)

96

##### Write Users to CSV File

In [14]:
import csv
csv_file = './data/output.csv'
field_names:list = None
results = um.advanced_search(f"accountid:{gis.properties.id}", as_dict=True,max_users=-1)
field_names = list(results.get("results")[0].keys()) + ['description']
with open(csv_file, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=field_names)
    writer.writeheader()
    for user in results['results']:
        writer.writerow(user)     

#### List User's Groups

In [15]:
um.user_groups(um.search("dev*")[:2])

[{'username': 'andrew57',
  'total': 67,
  'groups': [{'id': '019671d6889641caaa62666df9b35740',
    'title': 'parntered collab test'},
   {'id': '03871c826cff466fa6015f1163677830', 'title': 'Shared Update Group'},
   {'id': '09a73a911ea843cea083cfe70545ecf6', 'title': 'sdfgsdfgsdfh Content'},
   {'id': '0c04228089a243149a503922ae17901c',
    'title': 'sdfgsdfg343gsdfghsdfhash Core Team'},
   {'id': '0c27e80243984ae2946c4f43ceb94115', 'title': 'Name'}]},
 {'username': 'arcgis_python',
  'total': 58,
  'groups': [{'id': '00df55a8a6324c5aa4fc946d01aebb01',
    'title': 'Gallery',
    'thumbnail': 'Gallery.png'},
   {'id': '0352e4178bb24751847ef9798abb6976', 'title': 'Agriculturey'},
   {'id': '03587dc5f53b45669448b873d3fdcf43',
    'title': 'Network Operations',
    'thumbnail': 'NetworkOperations.png'},
   {'id': '0d753a8388a74d8798b47db800f8023e',
    'title': 'Site from Intro NB API Content'},
   {'id': '10753adc0d4a4649b497b2fb28ad6483',
    'title': 'demo-workforce-project'}]}]

#### Access a User's Items

- At 2.4.1+ this returns a Generator
- A generator is a special type of function or expression that creates an iterator, allowing for the generation of values on demand instead of constructing an entire sequence in memory at once. 

In [16]:
users[0].items()

<generator object User.items at 0x178ef8940>

In [17]:
for item in users[0].items():
    print(item)
    break

<Item title:"site2Apr4" type:Hub Site Application owner:geosaurus_hub>


### Creating New Users

In [18]:
# Check the user types available
um.counts('user_type')

,key,count
0,advancedUT,13
1,creatorUT,71
2,GISProfessionalAdvUT,10
3,GISProfessionalStdUT,2


In [19]:
import uuid
username = f"RUser{uuid.uuid4().hex[:4]}"
password = f"!{uuid.uuid4().hex[:8]}A"
um = gis.users
new_user = um.create(username=username, password=password, 
                     firstname="Johnny", lastname="Human", 
                     email='testsadf@esri.com', 
                     role="org_publisher")
new_user

<User username:RUser6271>

username, password

In [20]:
new_password = f"!{uuid.uuid4().hex[:8]}A"

#### Reset the Password

In [21]:
new_user.reset(
    password=password,
    new_password=new_password,
    new_security_question=1,
    new_security_answer=uuid.uuid4().hex[:10],
    reset_by_email=False,
)

True

In [22]:
GIS(username=username, 
    password=new_password, 
    verify_cert=False).users.me

Setting `verify_cert` to False is a security risk, use at your own risk.


<User username:RUser6271>

#### Deleting the User

In [23]:
new_user.delete()

True

### Working with Roles and User Types

#### User types

- User type determines the privileges that can be granted to the member through a default or custom role
- Common Roles:
  + viewer, creator and administrator

In [24]:
user = gis.users.me

In [25]:
user.user_types()['id']

'GISProfessionalAdvUT'

In [26]:
user.update_license_type("creatorUT")

True

In [27]:
user.user_types()['id']

'creatorUT'

In [28]:
user.update_license_type("GISProfessionalAdvUT")
user.user_types()['id']

'GISProfessionalAdvUT'

#### Working with Roles

- A role defines the set of privileges assigned to a member

**Accessing Role Manager**

In [29]:
rm = gis.users.roles
rm

**Listing Roles**

In [30]:
rm.all()

[<Role name: Viewer, description: Viewer>,
 <Role name: Data Editor, description: Data Editor>,
 <Role name: Facilitator, description: Facilitator>,
 <Role name: role_b3721, description: description>,
 <Role name: role_b35ff, description: description>,
 <Role name: role_7ea92, description: description>,
 <Role name: role_b74eb, description: description>,
 <Role name: role_52997, description: description>,
 <Role name: role_569c6, description: description>,
 <Role name: role_3819a, description: description>,
 <Role name: role_f07f2, description: description>,
 <Role name: role_7a57a, description: description>,
 <Role name: role_55dae, description: description>,
 <Role name: role_ceb8a, description: description>,
 <Role name: role_c9cb9, description: description>,
 <Role name: role_eb600, description: description>,
 <Role name: role_f2e02, description: description>,
 <Role name: role_5e937, description: description>,
 <Role name: role_7c854, description: description>,
 <Role name: role_f

**Check for Existence of a Role**

In [31]:
rm.exists('DataEditorRole')

False

In [32]:
role = rm.create(name="DataEditorRole", 
                 description="Allow to modify service data", 
                 privileges=[
                        "features:user:edit",
                        "features:user:fullEdit",
                        "opendata:user:designateGroup",
                        "portal:admin:viewUsers",
                        "portal:user:createGroup"]
                )
role

<Role name: DataEditorRole, description: Allow to modify service data>

**Removing the Role**

In [33]:
role.delete()

True

## Managing Content

In [34]:
cm = gis.content
cm

### Working with Content

**The content manager allows users and administrators to work with, find and manage content**

#### Searching

##### `search` Example

- provides a simple search method
- max items is 10,000
- do not have full control over searches

In [35]:
cm.search(query="title: battle", item_type="Feature Layer", outside_org=False)

[]

In [36]:
cm.search(query="title: battle", item_type="Feature Layer", outside_org=True)

[<Item title:"Civil_War_Battles" type:Feature Layer Collection owner:t3g09_BarbareeDuke>,
 <Item title:"HLL_Battle_App___Front_Lines_HF" type:Feature Layer Collection owner:topher303>,
 <Item title:"Density Based Clustering Time _10_ 1__WFL1" type:Feature Layer Collection owner:mcovey1_GISandData>,
 <Item title:"ACLED Nigeria Event Layers" type:Feature Layer Collection owner:K1060493@NZDF.MIL.NZ_nzdf>,
 <Item title:"MS_USH_IM_T09_L05_001M_N view" type:Feature Layer Collection owner:skktracy>,
 <Item title:"Last day Battle of the Alamo _1836__ 3D_WSL3" type:Feature Layer Collection owner:157_275_1923_9_byui>,
 <Item title:"1835_TXRev_Interactive_Webmap_WFL1" type:Feature Layer Collection owner:lynnette.cen@glo.texas.gov>,
 <Item title:"Key US Civil War Battles" type:Feature Layer Collection owner:Esri_GeoInquiry_History>,
 <Item title:"5kmNearest_Site_Poteniels_to_Battles" type:Feature Layer Collection owner:LroyCWT>,
 <Item title:"Civil_War_Battles - Battles_1861" type:Feature Layer Co

##### `advanced_search` Example

- full control searching option
- removed limitations of `search`
- returns items as dictionary, which speeds up searches
- leverage system for simple statistics about content

**How Many Item to Examine?**

In [37]:
count = cm.advanced_search('title: battle AND  (type:"feature service")', return_count=True)
count

985

In [38]:
items = cm.advanced_search('title: battle AND  (type:"feature service")', max_items=count, 
                           sort_field='avgRating', sort_order='desc')['results']
items[10:20]

[<Item title:"1861 Civil War Battles" type:Feature Layer Collection owner:GaDOE_SocialStudies_K-5_Atlas>,
 <Item title:"Shenandoah Valley Battlefields" type:Feature Layer Collection owner:allison.tillett_vdcr>,
 <Item title:"Civil War - All Battles as Hotspot Map (1861-1865)" type:Feature Layer Collection owner:BucknellGIS>,
 <Item title:"CivilWarP2A" type:Feature Layer Collection owner:chris.bun_madisonschools>,
 <Item title:"Civil War Battles Won by North" type:Feature Layer Collection owner:BucknellGIS>,
 <Item title:"ukraine battles 2022 Wk_12345" type:Feature Layer Collection owner:KHANW6@farmingdale.edu_FarmingdaleSC>,
 <Item title:"Predator Control Map_WFL1" type:Feature Layer Collection owner:ENMbiodiversity>,
 <Item title:"Nigeria ID change_WFL1" type:Feature Layer Collection owner:jacksontx2013>,
 <Item title:"UsHist_Virginia_features" type:Feature Layer Collection owner:Maps.com_carto>,
 <Item title:"Summarizes_Key_US_Civil_War_Battles_in_Cataloged_Desertion_Materials" type:

**Gathering Information from Searches**

- In this demo we will see how much new content was added to the organization in the last 5 days.

In [39]:
import datetime as _dt

now =_dt.datetime.now(_dt.timezone.utc)
then = now - _dt.timedelta(days=5)


In [40]:
cm.advanced_search(
    f"created: [{int(then.timestamp()* 1000)} TO {int(now.timestamp()* 1000)}] AND accountid:{gis.properties.id}", 
    return_count=True)

61

#### Adding and Publishing Content

In [41]:
import uuid
username = f"FedUser{uuid.uuid4().hex[:3]}"
password = f"!{uuid.uuid4().hex[:6]}A"
user = gis.users.create(username=username,
        password=password,
        firstname=uuid.uuid4().hex[:6],
        lastname=uuid.uuid4().hex[:6],
        email=uuid.uuid4().hex[:6] + "@esri.com",
        role='org_publisher')
user

<User username:FedUser778>

In [42]:
list(user.items())

[]

##### Publishing a Table

In this scenerio, the Administrator is going to add and publish a table to the newly created user from the previous steps.

In [43]:
import io, uuid
import pandas as pd
from arcgis.gis import ItemProperties, ItemTypeEnum

###### Load the Table into Memory

In [44]:
buffer = io.StringIO()
df = pd.read_csv("./data/banklist.csv")
df

,Bank Name,City,ST,CERT,Acquiring Institution,Closing Date,Updated Date
0,Fayette County Bank,Saint Elmo,IL,1802,"United Fidelity Bank, fsb",26-May-17,26-Jul-17
1,"Guaranty Bank, (d/b/a BestBank in Georgia & Mi...",Milwaukee,WI,30003,First-Citizens Bank & Trust Company,5-May-17,26-Jul-17
2,First NBC Bank,New Orleans,LA,58302,Whitney Bank,28-Apr-17,26-Jul-17
3,Proficio Bank,Cottonwood Heights,UT,35495,Cache Valley Bank,3-Mar-17,18-May-17
4,Seaway Bank and Trust Company,Chicago,IL,19328,State Bank of Texas,27-Jan-17,18-May-17
...,...,...,...,...,...,...,...
548,"Superior Bank, FSB",Hinsdale,IL,32646,"Superior Federal, FSB",27-Jul-01,19-Aug-14
549,Malta National Bank,Malta,OH,6629,North Valley Bank,3-May-01,18-Nov-02
550,First Alliance Bank & Trust Co.,Manchester,NH,34264,Southern New Hampshire Bank & Trust,2-Feb-01,18-Feb-03
551,National State Bank of Metropolis,Metropolis,IL,3815,Banterra Bank of Marion,14-Dec-00,17-Mar-05


In [45]:
df.to_csv(buffer)

###### Get the Destination User's Root Folder

In [46]:
folder = gis.content.folders.get(owner=user)
folder

< Folder: Root Folder Owner: FedUser778>

###### Populate the Item Properties and Add the Content

In [47]:
ip = ItemProperties(item_type=ItemTypeEnum.CSV, 
                    title='Failed Banks', 
                    file_name=f"failedbanks{uuid.uuid4().hex[:5]}.csv")
ip

<ItemProperties: title=Failed Banks, type=ItemTypeEnum.CSV>

In [48]:
result = folder.add(item_properties=ip, file=buffer)
result

< Job for Item: d0e75ca89c9742628e3b13fd5a422e18 >

In [49]:
item = result.result()
item

<Item title:"Failed Banks" type:CSV owner:FedUser778>

###### Analyze the Item and Publish the Table

In [50]:
analyzed = gis.content.analyze(item=item)
publish_parameters = analyzed['publishParameters']
publish_parameters[
        'name'
    ] = f"Failed_Banks_{uuid.uuid4().hex[:2]}"  # this needs to be updated
publish_parameters['locationType'] = None  # this makes it a hosted table
published_item = item.publish(publish_parameters)

In [51]:
published_item

<Item title:"Failed Banks" type:Table Layer owner:FedUser778>

In [52]:
published_item.delete(permanent=True)
item.delete(permanent=True)

True

## Viewing All Content

- Administrators have  the ability to view **ALL** content on a given organization from the `gis.admin` endpoint

In [53]:
# View all Feature Services in a Given Organization
for item in gis.admin.content(item_type=ItemTypeEnum.FEATURE_SERVICE,
                           sort_field='created',
                           order='desc'):
    print(item)

<Item title:"calculate_density_point_data_885d" type:Table Layer owner:arcgis_python>
<Item title:"restaurants_123" type:Table Layer owner:arcgis_python>
<Item title:"aggregate_points_point_data_0f8f" type:Table Layer owner:arcgis_python>
<Item title:"aggregate_points_polygon_data_b87d" type:Feature Layer Collection owner:arcgis_python>
<Item title:"aggregate_points_point_data_8404" type:Table Layer owner:arcgis_python>
<Item title:"aggregate_points_polygon_data_f885" type:Feature Layer Collection owner:arcgis_python>
<Item title:"aggregate_points_point_data_90d8" type:Table Layer owner:arcgis_python>
<Item title:"aggregate_points_polygon_data_f29d" type:Feature Layer Collection owner:arcgis_python>
<Item title:"add_to_def_fl_e4b42" type:Feature Layer Collection owner:arcgis_python>
<Item title:"california_cities_6154" type:Feature Layer Collection owner:arcgis_python>
<Item title:"create_drive_time_areas_office_data_1070" type:Feature Layer Collection owner:arcgis_python>
<Item title:

KeyboardInterrupt: 

## Metadata 

- Administrators can enable metadata for the organization

In [54]:
mm = gis.admin.metadata
mm

< MetadataManager at https://geosaurus.maps.arcgis.com >

In [55]:
mm.enable()

True

In [56]:
mm.is_enabled

True